In [ ]:
from bs4 import BeautifulSoup
import datetime
import pandas as pd
from pandas import ExcelWriter
from selenium import webdriver
from time import sleep
import os
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select
from random import uniform
from selenium.common.exceptions import TimeoutException

print("MT MFSA Web Scraping Tool v.1.0")

# Assigning current time, output file name and ExcelWriter object
now = datetime.datetime.now()
filename = 'MT MFSA SQL Ready {}.xlsx'.format(str(now).replace(":", ".")[:-7])
writer = ExcelWriter(filename)

# Assigning the folders that are going to be used in the process
scriptfolder = os.path.dirname(os.path.abspath(__file__))
tempfolder = os.path.join(scriptfolder, 'tempfolder')
os.chdir(scriptfolder)
outputfolder = r'D:\Regulators\output\ready'

# Creating tempfolder if it doesn't exists, emptying in if it does exist
if os.path.exists(tempfolder):
    for temp_file in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, temp_file))
else:
    os.mkdir(tempfolder)

# Starting Chrome driver, set to download files in tempfolder
chromeOptions = webdriver.ChromeOptions()
prefs = {"plugins.always_open_pdf_externally": True,
         "download.prompt_for_download": False,
         "download.default_directory": tempfolder}
chromeOptions.add_experimental_option("prefs", prefs)
chromeOptions.add_argument("--no-sandbox")
chromeOptions.add_argument("--disable-dev-shm-usage")
chromeOptions.add_argument("--disable-gpu")
driver = webdriver.Chrome(options=chromeOptions)

# Creating dictionary with Regcodes and their respective URLs
regdict = {'MT MFSA 1': ('https://www.mfsa.mt/financial-services-register/', 'License Holders')}

# Creating dictionary to contain regulators data and then be converted to a pandas DataFrame
sqldict = {'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [],
           'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
           'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [],
           'Address_1': [], 'Address_2': [], 'City': [],
           'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [],
           'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
           'RegCtry': [], 'RegCode': [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [],
           'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
           'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [],
           'Zip - Mother company': [], 'Cntry - Mother company': [],
           'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

skipped_views = 0
skipped_status_counter = 0  # global counter for skipped views due to status

def get_with_retries(driver, url, retries=5, base_delay=30):
    for attempt in range(retries):
        try:
            driver.get(url)
            return
        except Exception as e:
            wait_time = base_delay * (2 ** attempt) + uniform(0, 3)
            print(f"Attempt {attempt+1} failed: {e}. Waiting {wait_time:.2f} seconds before retrying.")
            sleep(wait_time)
    raise Exception("Unable to load URL after multiple attempts")

def reload_on_error(driver, url, wait_seconds=30):
    print(f"Encountered an error. Waiting {wait_seconds} seconds before reloading...")
    sleep(wait_seconds)
    get_with_retries(driver, url)

AuthorisedPersonBasicDetails = [
    'Authorised Person Name', 'MBR Registration Code', 'Authorised Person ID',
    'Company Registration Date', 'LEI Code', 'Registered Address', 'Phone',
    'Email', 'Website', 'Remarks', 'Authorisation', 'Status', 'Authorisation Issue Date'
]

records = []  # list to store each record as a dictionary

def select_parent_option(driver, parent, parent_values, max_attempts=3):
    """Helper function to select parent option with retries"""
    for attempt in range(max_attempts):
        try:
            # Ensure we're in the iframe
            driver.switch_to.default_content()
            WebDriverWait(driver, 30).until(
                EC.frame_to_be_available_and_switch_to_it((By.XPATH, "//iframe[@title='Financial Service Register']"))
            )

            # Get fresh reference to parent dropdown
            parent_dropdown_elem = WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.ID, "parentLicenceTypes"))
            )

            # Use JavaScript to select the parent
            if parent in parent_values:
                driver.execute_script("""
                    arguments[0].value = arguments[1];
                    let event = new Event('change', { bubbles: true });
                    arguments[0].dispatchEvent(event);
                """, parent_dropdown_elem, parent_values[parent])
                sleep(5)

                # Verify selection
                selected_value = driver.execute_script("return arguments[0].value;", parent_dropdown_elem)
                if (selected_value == parent_values[parent]):
                    print(f"Successfully selected parent '{parent}' with value '{parent_values[parent]}'")
                    return True, parent_dropdown_elem

            if attempt < max_attempts - 1:
                print(f"Retrying parent selection for '{parent}' (attempt {attempt + 1})")
                driver.refresh()
                sleep(5)

        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {str(e)}")
            if attempt < max_attempts - 1:
                print("Refreshing page and retrying...")
                driver.refresh()
                sleep(5)

    print(f"Failed to select parent '{parent}' after {max_attempts} attempts")
    return False, None

def process_view_form(driver, form_index, total_forms, parent, sub, parent_values):
    """Process a single view form using explicit waits and retries."""
    max_attempts = 3
    global skipped_status_counter
    for attempt in range(max_attempts):
        try:
            print(f"Processing view form {form_index+1} of {total_forms} (attempt {attempt+1})")
            # Increase wait time to 30 seconds for main results
            main_results = WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.ID, "mainResults"))
            )
            view_forms = main_results.find_elements(By.TAG_NAME, "form")
            if form_index >= len(view_forms):
                print(f"Form index {form_index} out of range.")
                return False
            current_form = view_forms[form_index]
            driver.execute_script("arguments[0].scrollIntoView(true);", current_form)
            sleep(5)

            # Wait up to 20 seconds for the view button to be clickable
            view_button = WebDriverWait(current_form, 30).until(
                EC.element_to_be_clickable((By.TAG_NAME, "button"))
            )
            driver.execute_script("arguments[0].click();", view_button)
            sleep(uniform(5, 7))

            # Wait up to 30 seconds for details to load
            detail_html = WebDriverWait(driver, 30).until(lambda d: d.page_source)
            detail_soup = BeautifulSoup(detail_html, "html.parser")

            # Check licence status before extracting details
            status_value = None
            for row in detail_soup.find_all("div", class_="row"):
                label_tag = row.find("label")
                if label_tag and "Status:" in label_tag.get_text(strip=True):
                    sibling = row.find("div", class_=lambda x: x and "col-9" in x)
                    if sibling:
                        status_value = sibling.get_text(strip=True)
                    break

            if status_value and ("Surrendered" in status_value or "Suspended" in status_value or "Revoked" in status_value):
                print("Licence status is surrendered/suspended; skipping this view option.")
                skipped_status_counter += 1  # increment skip counter
            else:
                details = {}
                for row in detail_soup.find_all("div", class_="row"):
                    label_tag = row.find("label")
                    value_tag = row.find("div", class_=lambda x: x and ("col-8" in x or "col-9" in x))
                    if label_tag and value_tag:
                        label = label_tag.get_text(strip=True).rstrip(":")
                        value = value_tag.get_text(strip=True)
                        details[label] = value

                        sqldict['Name'].append(details.get("Authorised Person Name", ""))
                        internal1 = details.get("MBR Registration Code", "")
                        if internal1 == "NOT IN ROC" or internal1 == "NOT IN MBR":
                            internal1 = ""
                        sqldict['InternalID_1'].append(internal1)
                        sqldict['InternalID_1_type'].append("MBR Registration Code" if internal1 else "")
                        internal2 = details.get("Authorised Person ID", "")
                        sqldict['InternalID_2'].append(internal2)
                        sqldict['InternalID_2_type'].append("Authorised Person ID" if internal2 else "")
                        sqldict['Address_1'].append(details.get("Registered Address", ""))
                        sqldict['Phone'].append(details.get("Phone", ""))
                        sqldict['Email'].append(details.get("Email", ""))
                        sqldict['Website'].append(details.get("Website", ""))
                        sqldict['ListProcessDate'].append(processdate)
                        sqldict['RegCtry'].append('MT')
                        sqldict['RegCode'].append('MFSA')
                        sqldict['ListCode'].append(reg.split(' ')[-1])
                        sqldict['RegulationType'].append("Regulated")
                        sqldict['ListName'].append(list_name)
                        sqldict["RegulationDate"].append(details.get("Authorisation Issue Date", ""))
                        sqldict["LEI Code"].append(details.get("LEI Code", ""))



            # Return to the listing page (simulate pressing the browser back button)
            try:
                driver.back()
            except Exception as e:
                print("driver.back() failed: " + str(e))
            sleep(5)  # Allow the page to go back

            # Ensure we have switched to the correct iframe and that the listing is visible
            driver.switch_to.default_content()
            WebDriverWait(driver, 30).until(
                EC.frame_to_be_available_and_switch_to_it((By.XPATH, "//iframe[@title='Financial Service Register']"))
            )
            WebDriverWait(driver, 30).until(
                EC.presence_of_element_located((By.ID, "mainResults"))
            )

            # Reselect parent and sub dropdowns with retries
            reselect_parent = False
            for r in range(3):
                success, _ = select_parent_option(driver, parent, parent_values)
                if success:
                    reselect_parent = True
                    break
                sleep(5)
            if not reselect_parent:
                raise Exception("Could not reselect parent after view processing.")

            WebDriverWait(driver, 30).until(
                lambda d: len(Select(d.find_element(By.ID, "subLicenceTypes")).options) > 1
            )
            sub_dropdown = Select(driver.find_element(By.ID, "subLicenceTypes"))
            sub_dropdown.select_by_visible_text(sub)
            sleep(5)
            return True

        except Exception as e:
            err_msg = str(e).lower()
            print(f"Attempt {attempt+1} failed for view form {form_index+1}: {str(e)}")
            # Option 1: If renderer crash/error is detected, reinitialize the driver
            if "tab crashed" in err_msg or "bad ipc" in err_msg:
                print("Renderer crash detected. Reinitializing driver (Option 1).")
                try:
                    driver.quit()
                except Exception as quit_ex:
                    print("Driver quit failed: " + str(quit_ex))
                # Reinitialize the driver using your chromeOptions
                driver = webdriver.Chrome(options=chromeOptions)
                # Optionally, navigate back to the target URL and reenter the iframe context.
                # If your URL is stored (e.g. in 'url'), use:
                get_with_retries(driver, url)
                sleep(5)
                try:
                    driver.switch_to.default_content()
                    WebDriverWait(driver, 30).until(
                        EC.frame_to_be_available_and_switch_to_it((By.XPATH, "//iframe[@title='Financial Service Register']"))
                    )
                    success, _ = select_parent_option(driver, parent, parent_values)
                    if not success:
                        raise Exception("Reselecting parent failed on driver reinitialization.")
                    WebDriverWait(driver, 30).until(
                        lambda d: len(Select(d.find_element(By.ID, "subLicenceTypes")).options) > 1
                    )
                    sub_dropdown = Select(driver.find_element(By.ID, "subLicenceTypes"))
                    sub_dropdown.select_by_visible_text(sub)
                    sleep(5)
                except Exception as re_ex:
                    print("Driver reinitialization failed: " + str(re_ex))
                return False

            if attempt < max_attempts - 1:
                print("Refreshing page and retrying view processing...")
                driver.refresh()
                sleep(5)
                try:
                    driver.switch_to.default_content()
                    WebDriverWait(driver, 30).until(
                        EC.frame_to_be_available_and_switch_to_it((By.XPATH, "//iframe[@title='Financial Service Register']"))
                    )
                    success, _ = select_parent_option(driver, parent, parent_values)
                    if not success:
                        raise Exception("Reselecting parent failed on retry.")
                    WebDriverWait(driver, 30).until(
                        lambda d: len(Select(d.find_element(By.ID, "subLicenceTypes")).options) > 1
                    )
                    sub_dropdown = Select(driver.find_element(By.ID, "subLicenceTypes"))
                    sub_dropdown.select_by_visible_text(sub)
                    sleep(5)
                except Exception as re_ex:
                    print(f"Retry reselection failed: {str(re_ex)}")
            else:
                print(f"Failed to process view form {form_index+1} after {max_attempts} attempts.")
                return False
    return False

def process_parent_option(driver, parent, parent_values):
    """Process all sub-options for a given parent using removal-based logic for view forms."""
    success, parent_elem = select_parent_option(driver, parent, parent_values)
    if not success:
        print(f"Failed to select parent '{parent}'")
        return False

    try:
        # Wait for and get sub dropdown options
        WebDriverWait(driver, 30).until(
            lambda d: len(Select(d.find_element(By.ID, "subLicenceTypes")).options) > 1
        )
        # Create a local copy of sub option values
        sub_dropdown = Select(driver.find_element(By.ID, "subLicenceTypes"))
        sub_texts = [option.text for option in sub_dropdown.options if option.get_attribute("value") != "0"]
        print(f"Found {len(sub_texts)} sub options for {parent}")

        # Process each sub option
        for sub in sub_texts:
            print(f"Processing sub option: {sub}")
            try:
                # Reinitialize the sub-dropdown element before selecting to avoid stale elements
                sub_dropdown_elem = WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.ID, "subLicenceTypes"))
                )
                sub_dropdown = Select(sub_dropdown_elem)
                sub_dropdown.select_by_visible_text(sub)
                sleep(uniform(5, 7))

                # Get initial view forms count
                main_results = WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.ID, "mainResults"))
                )
                view_forms = main_results.find_elements(By.TAG_NAME, "form")
                total_forms = len(view_forms)
                print(f"Found {total_forms} view forms under {parent} / {sub}")

                # Process each view form
                form_index = 0
                while form_index < total_forms:
                    if process_view_form(driver, form_index, total_forms, parent, sub, parent_values):
                        form_index += 1
                    else:
                        print(f"Skipping problematic view form {form_index+1}")
                        form_index += 1

                # Inside your sub option processing block:
                print(f"Found view forms under {parent} / {sub}")

                # Get an initial count of view forms
                main_results = WebDriverWait(driver, 30).until(
                    EC.presence_of_element_located((By.ID, "mainResults"))
                )
                view_forms = main_results.find_elements(By.TAG_NAME, "form")
                previous_count = len(view_forms)

                while True:
                    # Re-read the current view forms count
                    main_results = WebDriverWait(driver, 30).until(
                        EC.presence_of_element_located((By.ID, "mainResults"))
                    )
                    view_forms = main_results.find_elements(By.TAG_NAME, "form")
                    current_count = len(view_forms)
                    print(f"Current view forms count under {parent} / {sub}: {current_count}")

# If no forms remain, break out of the loop
                    # If no forms remain, break out of the loop
                    if current_count == 0:
                        print(f"No view forms remain under {parent} / {sub}")
                        break

                    # If the count has not decreased compared to the previous iteration, assume processing is done
                    if previous_count is not None and current_count >= previous_count:
                        print("View forms count did not decrease; assuming no unprocessed forms remain.")
                        break

                    # Update previous_count for next iteration
                    previous_count = current_count

                    # Process the first view form (index 0) from the current list
                    print(f"Processing view form 1 of {current_count} under {parent} / {sub}")
                    if process_view_form(driver, 0, current_count, parent, sub, parent_values):
                        print("View form processed successfully.")
                    else:
                        print("Skipping problematic view form.")

                    sleep(uniform(5, 7))  # brief wait before checking the count again

            except Exception as e:
                print(f"Error processing sub option '{sub}': {str(e)}")
                continue

        return True

    except Exception as e:
        print(f"Error processing parent '{parent}': {str(e)}")
        return False

def get_parent_values(driver):
    """Gets the mapping of parent names to their values from the dropdown"""
    parent_dropdown = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.ID, "parentLicenceTypes"))
    )
    return {
        option.text.strip(): option.get_attribute("value")
        for option in Select(parent_dropdown).options
        if option.get_attribute("value") != "0"
    }

# For each regulator URL, process only the first parent option for testing
for reg, (url, list_name) in regdict.items():
    print("Working with {}".format(reg))
    get_with_retries(driver, url)
    sleep(uniform(5, 7))

    # Switch to iframe and get parent dropdown options
    WebDriverWait(driver, 30).until(
        EC.frame_to_be_available_and_switch_to_it((By.XPATH, "//iframe[@title='Financial Service Register']"))
    )
    print("Switched to Financial Service Register iframe.")

    # Get parent dropdown element and its options
    parent_dropdown_elem = WebDriverWait(driver, 30).until(
        EC.presence_of_element_located((By.ID, "parentLicenceTypes"))
    )
    parent_dropdown = Select(parent_dropdown_elem)
    parent_texts = [option.text for option in parent_dropdown.options if option.get_attribute("value") != "0"]
    print("Found {} parent options.".format(len(parent_dropdown.options)))
    if not parent_texts:
        print("No valid parent options found.")
        driver.quit()

    # Get parent values mapping
    parent_values = get_parent_values(driver)
    print("Found parent values:", parent_values)

    # Process all parents
    for parent in parent_texts:
    # MAX_PARENTS = 7
    # for i, parent in enumerate(parent_texts):
    #     if i > MAX_PARENTS:
    #         print(f"Reached maximum parent limit of {MAX_PARENTS}. Stopping further processing.")
    #         break
        print("\nProcessing parent option: {}".format(parent))
        if not process_parent_option(driver, parent, parent_values):
         print("Error processing parent '{}'".format(parent))
        else:
         print("Finished processing parent '{}'".format(parent))

    # # Process parent options starting from index 7 until the end
    # for i, parent in enumerate(parent_texts[7:], start=7):
    #     print("\nProcessing parent option: {}".format(parent))
    #     if not process_parent_option(driver, parent, parent_values):
    #         print("Error processing parent '{}'".format(parent))
    #     else:
    #         print("Finished processing parent '{}'".format(parent))


# Create the DataFrame from your sqlDict
df = pd.DataFrame.from_dict(sqldict, orient='index').transpose()

# Define key columns to group by (adjust as needed)
group_cols = ['Name', 'InternalID_1', 'InternalID_2']

# Determine which columns to consider for "data filled" (exclude grouping keys)
data_cols = [col for col in df.columns if col not in group_cols]

# Create a helper column 'filled' that counts non-empty cells in data_cols for each row.
df['filled'] = df[data_cols].apply(
    lambda r: r.astype(str).str.strip().replace("", pd.NA).notna().sum(),
    axis=1
)

# Create a helper column 'id_priority' that is 1 if either InternalID_1 or InternalID_2 is non-empty, else 0.
df['id_priority'] = df.apply(
    lambda row: 1 if ((isinstance(row['InternalID_1'], str) and row['InternalID_1'].strip())
                      or (isinstance(row['InternalID_2'], str) and row['InternalID_2'].strip()))
                else 0,
    axis=1
)

# Sort by 'Name', then descending by 'id_priority' and 'filled' so that rows with an ID and more filled data come first.
df_sorted = df.sort_values(['Name', 'id_priority', 'filled'], ascending=[True, False, False])

# Drop duplicates based solely on the 'Name' column keeping the first (best) record.
df_dedup = df_sorted.drop_duplicates(subset=['Name'], keep='first')

# Drop the helper columns before exporting
df_dedup = df_dedup.drop(columns=['filled', 'id_priority'])

# Write the deduplicated DataFrame to Excel
df_dedup.to_excel(excel_writer=writer, sheet_name='SQL Ready', index=False)
writer.close()
driver.quit()

# Finally, print the counters:
print(f"Number of view forms skipped due to status (surrendered/suspended/revoked): {skipped_status_counter}")
print(f"Number of duplicate rows deleted: {df.shape[0] - df_dedup.shape[0]}")




    
    
    
    